# 11 — Project a direction onto Goodfire's Llama 3.3 70B SAE features

Given any direction in Llama 3.3 70B's residual stream at layer 50, this notebook ranks the SAE features that direction is most aligned with. Output: top-K features by cosine, with Neuronpedia URLs and (optionally) auto-interp descriptions.

**No model loading required.** We only need the SAE weights (4.3 GB download from HF) and the direction vector itself. Runs on a laptop.

**SAE source**: `Goodfire/Llama-3.3-70B-Instruct-SAE-l50` — single `.pt` file, TopK SAE, L0=121, trained on LMSYS-Chat-1M activations at layer 50. Browseable on Neuronpedia at `https://www.neuronpedia.org/llama3.3-70b/50-...` (path depends on Goodfire's neuronpedia model ID).

**Direction sources** (any one):
1. Local: our exp10 `directions.npz` for `llama33_70b` (after running notebook 10).
2. HF download: pre-computed assistant axis from `lu-christina/assistant-axis-vectors/llama-3.3-70b/assistant_axis.pt`.
3. HF download: refusal direction from `huihui-ai/Llama-3.3-70B-Instruct-abliterated/refusal_dir.pth`.

Pick one in the config cell. Notebook 12 does the multi-direction comparison.

In [ ]:
# Install (Colab / fresh box). Comment out if already in env.
# !pip install -q huggingface_hub torch numpy matplotlib requests pyyaml

In [ ]:
# ---- Config ----
from pathlib import Path

# Which direction to project? Set ONE of these:
DIRECTION_SOURCE = 'assistant_axis'   # 'sui_local' | 'assistant_axis' | 'refusal'

# For 'sui_local' — path to our exp10 directions.npz and which key to pull.
SUI_NPZ_PATH    = Path.cwd().parent / 'exp06_lamma' / 'directions.npz'
SUI_KEY         = 'mm_dir__response_last__layer_050'   # MM at the SAE's layer (50)

# Goodfire SAE
SAE_REPO_ID     = 'Goodfire/Llama-3.3-70B-Instruct-SAE-l50'
SAE_FILENAME    = 'Llama-3.3-70B-Instruct-SAE-l50.pt'
SAE_LAYER       = 50

# How many features to surface per side
TOP_K           = 25

# Optional: query Neuronpedia API for auto-interp descriptions (slower; rate-limited)
FETCH_AUTOINTERP = False
NEURONPEDIA_MODEL_SLUG = 'llama3.3-70b'   # confirm on neuronpedia.org/llama3.3-70b
NEURONPEDIA_SAE_SLUG   = '50-goodfire-sae-l50'  # placeholder; check actual Neuronpedia ID

print(f'direction src : {DIRECTION_SOURCE}')
print(f'SAE           : {SAE_REPO_ID} (L{SAE_LAYER})')
print(f'top-K         : {TOP_K}')

### Method note — why decoder cosine, not encoder.forward(direction)

Each decoder row `W_dec[i]` is the residual-stream "write direction" of feature *i*. So `cosine(direction, W_dec[i])` answers: *"if feature i fires, how much does it push the residual stream along this direction?"* — which is the standard interpretive lens used by SAELens, the Goodfire research blog, and the AlignmentForum refusal/sycophancy work.

We deliberately **don't** just run `z = TopK(W_enc @ direction + b_enc)` and read off the active features. Mayne et al. (NeurIPS 2024, *"Can SAEs be used to decompose and interpret steering vectors?"*) showed three failure modes for that:

1. Steering directions are out-of-distribution for the SAE — their norms are far smaller than typical residual-stream activations, so the encoder bias dominates the pre-activations.
2. SAEs reconstruct with non-negative coefficients only, so they can't represent steering directions that have meaningful negative projections onto features.
3. With TopK SAEs (like Goodfire's, k=121) you'd just get the 121 largest pre-activations regardless of magnitude, hiding the OOD problem.

We rank by `W_dec` cosine as the primary signal and report `W_enc` cosine as a secondary check (they should mostly agree on sign-consistent SAEs — large disagreements are worth flagging).

## 1 — Download and load the Goodfire SAE

Goodfire's release is a single 4.3 GB `.pt` pickle. The internal layout isn't fully documented on the HF page, so the cell below loads, prints the structure, and tries common conventions. Adjust the W_enc / W_dec extraction if your version differs.

In [ ]:
import torch
from huggingface_hub import hf_hub_download

sae_path = hf_hub_download(repo_id=SAE_REPO_ID, filename=SAE_FILENAME)
print(f'downloaded: {sae_path}')

sae_obj = torch.load(sae_path, map_location='cpu', weights_only=False)
print(f'type      : {type(sae_obj).__name__}')

# Normalize to a state-dict-like mapping.
if hasattr(sae_obj, 'state_dict'):
    sd = sae_obj.state_dict()
elif isinstance(sae_obj, dict):
    sd = sae_obj
else:
    raise RuntimeError(f'Unexpected SAE object type {type(sae_obj)} — inspect manually.')

print(f'\nstate_dict keys (first 30):')
for k in list(sd.keys())[:30]:
    v = sd[k]
    shape = tuple(v.shape) if hasattr(v, 'shape') else type(v).__name__
    print(f'  {k:50s} {shape}')
if len(sd) > 30:
    print(f'  ... ({len(sd)} keys total)')

In [ ]:
# Extract encoder and decoder weights. Try common naming conventions.
# Goodfire's TopK SAEs typically expose: encoder.weight (d_sae, d_model),
# decoder.weight (d_model, d_sae), encoder.bias, decoder.bias.

def _find(state_dict, candidates):
    for c in candidates:
        if c in state_dict:
            return c, state_dict[c]
    raise KeyError(f'none of {candidates} found in state_dict; keys: {list(state_dict.keys())[:10]}...')

enc_key, W_enc = _find(sd, ['encoder.weight', 'W_enc', 'enc.weight', 'W_in'])
dec_key, W_dec = _find(sd, ['decoder.weight', 'W_dec', 'dec.weight', 'W_out'])

print(f'W_enc ({enc_key}): shape={tuple(W_enc.shape)}, dtype={W_enc.dtype}')
print(f'W_dec ({dec_key}): shape={tuple(W_dec.shape)}, dtype={W_dec.dtype}')

# Reorient both to (d_sae, d_model) so each row is one feature.
# Convention: nn.Linear stores weight as (out, in). For an encoder Linear(d_model -> d_sae)
# that gives (d_sae, d_model) already. For a decoder Linear(d_sae -> d_model) the stored
# weight is (d_model, d_sae), so transpose to put features on the row axis.
if W_enc.shape[0] < W_enc.shape[1]:
    print('  -> W_enc looks transposed; assuming first dim is d_sae')
if W_dec.shape[0] != W_enc.shape[0]:
    if W_dec.shape[1] == W_enc.shape[0]:
        print('  -> W_dec stored as (d_model, d_sae); transposing to (d_sae, d_model)')
        W_dec = W_dec.T
    else:
        raise RuntimeError(
            f'W_dec shape {tuple(W_dec.shape)} not compatible with W_enc {tuple(W_enc.shape)}'
        )

W_enc = W_enc.float()
W_dec = W_dec.float()
d_sae, d_model = W_enc.shape
assert W_dec.shape == (d_sae, d_model), f'W_dec must be ({d_sae},{d_model}), got {tuple(W_dec.shape)}'
print(f'\nd_sae={d_sae}, d_model={d_model}')
print(f'expansion = d_sae / d_model = {d_sae / d_model:.1f}x')

## 2 — Load the direction

Picks the source based on `DIRECTION_SOURCE`. All three end with a `(d_model,)` tensor in float32 on CPU.

In [ ]:
import numpy as np

def _load_sui_local(npz_path: Path, key: str) -> torch.Tensor:
    if not npz_path.exists():
        raise FileNotFoundError(
            f'no exp10 NPZ at {npz_path}. Run notebook 10 with MODEL_KEY="llama33_70b" first.'
        )
    arr = np.load(npz_path)
    if key not in arr.files:
        raise KeyError(f'key {key!r} not in NPZ. Available keys (first 10): {arr.files[:10]}')
    return torch.from_numpy(arr[key]).float()

def _load_assistant_axis() -> torch.Tensor:
    p = hf_hub_download(
        repo_id='lu-christina/assistant-axis-vectors',
        filename='llama-3.3-70b/assistant_axis.pt',
        repo_type='dataset',
    )
    obj = torch.load(p, map_location='cpu', weights_only=False)
    # File may be a tensor, a dict, or a (n_layers, d_model) tensor.
    if isinstance(obj, dict):
        # Heuristic: pick a layer-50 entry if present.
        for k in [50, '50', f'layer_{SAE_LAYER}', f'L{SAE_LAYER}']:
            if k in obj:
                return obj[k].float().squeeze()
        raise KeyError(f'assistant_axis.pt is a dict but no layer-{SAE_LAYER} key. Keys: {list(obj.keys())[:10]}')
    t = obj.float().squeeze()
    if t.ndim == 2:   # (n_layers, d_model) -> pick L50
        print(f'  axis is per-layer: shape={tuple(t.shape)}; picking row {SAE_LAYER}')
        return t[SAE_LAYER]
    return t

def _load_refusal() -> torch.Tensor:
    p = hf_hub_download(
        repo_id='huihui-ai/Llama-3.3-70B-Instruct-abliterated',
        filename='refusal_dir.pth',
    )
    obj = torch.load(p, map_location='cpu', weights_only=False)
    t = obj.float().squeeze() if torch.is_tensor(obj) else torch.tensor(obj).float()
    if t.ndim == 2:
        print(f'  refusal is per-layer: shape={tuple(t.shape)}; picking row {SAE_LAYER}')
        return t[SAE_LAYER]
    return t

if DIRECTION_SOURCE == 'sui_local':
    direction = _load_sui_local(SUI_NPZ_PATH, SUI_KEY)
elif DIRECTION_SOURCE == 'assistant_axis':
    direction = _load_assistant_axis()
elif DIRECTION_SOURCE == 'refusal':
    direction = _load_refusal()
else:
    raise ValueError(f'unknown DIRECTION_SOURCE={DIRECTION_SOURCE!r}')

direction = direction / (direction.norm() + 1e-10)   # unit-normalize
assert direction.shape == (d_model,), \
    f'direction shape {tuple(direction.shape)} != ({d_model},); SAE/direction d_model mismatch'
print(f'direction loaded: shape={tuple(direction.shape)}, norm=1.000 (post-normalize)')

## 3 — Project: cosine of direction with each feature's **decoder** row

`W_dec[i]` is the residual-stream direction feature *i* writes when it fires. So `cosine(direction, W_dec[i])` ranks features by how much they push the residual stream along our direction. Two views:

- **Top-K positive** — features that, when active, push *along* the direction (concept the direction encodes).
- **Top-K negative** — features that push *against* the direction (the opposing concept; informative because steering this way suppresses them).

The next cell also reports the top-K by encoder cosine for comparison; agreement = sign-consistent SAE behaving normally, large disagreement = worth a closer look.

In [ ]:
with torch.no_grad():
    dec_norms = W_dec.norm(dim=1) + 1e-10                  # (d_sae,)
    enc_norms = W_enc.norm(dim=1) + 1e-10                  # (d_sae,)
    sims_dec  = (W_dec @ direction) / dec_norms            # (d_sae,)  PRIMARY
    sims_enc  = (W_enc @ direction) / enc_norms            # (d_sae,)  secondary

# Save for later cells (Neuronpedia URLs, baseline, etc.)
sims = sims_dec
top_pos = torch.topk(sims_dec, TOP_K, largest=True)
top_neg = torch.topk(sims_dec, TOP_K, largest=False)

print(f'DECODER cosine sims: mean={sims_dec.mean():+.4f}  std={sims_dec.std():.4f}  '
      f'min={sims_dec.min():+.4f}  max={sims_dec.max():+.4f}')
print(f'ENCODER cosine sims: mean={sims_enc.mean():+.4f}  std={sims_enc.std():.4f}  '
      f'min={sims_enc.min():+.4f}  max={sims_enc.max():+.4f}')
print(f'corr(decoder, encoder) over all features = '
      f'{torch.corrcoef(torch.stack([sims_dec, sims_enc]))[0, 1]:+.4f}')

print(f'\nTop-{TOP_K} POSITIVE by DECODER cosine (features the direction promotes):')
for rank, (idx, sim) in enumerate(zip(top_pos.indices.tolist(), top_pos.values.tolist())):
    enc_c = sims_enc[idx].item()
    print(f'  #{rank+1:2d}  feature_{idx:6d}   cos_dec={sim:+.4f}   cos_enc={enc_c:+.4f}')

print(f'\nTop-{TOP_K} NEGATIVE by DECODER cosine (features the direction suppresses):')
for rank, (idx, sim) in enumerate(zip(top_neg.indices.tolist(), top_neg.values.tolist())):
    enc_c = sims_enc[idx].item()
    print(f'  #{rank+1:2d}  feature_{idx:6d}   cos_dec={sim:+.4f}   cos_enc={enc_c:+.4f}')

# Sanity: how much do encoder-top and decoder-top sets overlap?
top_dec_set = set(top_pos.indices.tolist())
top_enc_set = set(torch.topk(sims_enc, TOP_K, largest=True).indices.tolist())
overlap = len(top_dec_set & top_enc_set) / TOP_K
print(f'\nTop-{TOP_K} positive overlap (decoder vs encoder ranking): {overlap*100:.0f}%')

## 4 — Neuronpedia URLs

Generates one URL per top feature. Click through to read auto-interp descriptions and top-activating examples.

In [ ]:
def _np_url(feature_id: int) -> str:
    return f'https://www.neuronpedia.org/{NEURONPEDIA_MODEL_SLUG}/{NEURONPEDIA_SAE_SLUG}/{feature_id}'

print('POSITIVE side:')
for rank, (idx, sim) in enumerate(zip(top_pos.indices.tolist(), top_pos.values.tolist())):
    print(f'  cos={sim:+.4f}  {_np_url(idx)}')
print('\nNEGATIVE side:')
for rank, (idx, sim) in enumerate(zip(top_neg.indices.tolist(), top_neg.values.tolist())):
    print(f'  cos={sim:+.4f}  {_np_url(idx)}')

## 5 — (Optional) Fetch auto-interp descriptions via Neuronpedia API

Set `FETCH_AUTOINTERP = True` in the config cell. Public API; rate-limited so don't hammer it. If the model/SAE slugs are wrong you'll get 404s — confirm by visiting one of the URLs above first.

In [ ]:
if FETCH_AUTOINTERP:
    import requests, time
    def _fetch(feature_id):
        url = f'https://www.neuronpedia.org/api/feature/{NEURONPEDIA_MODEL_SLUG}/{NEURONPEDIA_SAE_SLUG}/{feature_id}'
        try:
            r = requests.get(url, timeout=10)
            if r.status_code != 200:
                return f'<HTTP {r.status_code}>'
            j = r.json()
            # Conventional Neuronpedia field; may vary by deployment.
            for k in ('explanation', 'description', 'autointerp'):
                if k in j and j[k]:
                    return j[k] if isinstance(j[k], str) else j[k].get('description', str(j[k]))[:200]
            return '<no explanation field>'
        except Exception as e:
            return f'<error: {e}>'
    print('POSITIVE side:')
    for idx, sim in zip(top_pos.indices.tolist(), top_pos.values.tolist()):
        desc = _fetch(idx)
        print(f'  feat {idx:6d}  cos={sim:+.3f}  {desc}')
        time.sleep(0.3)
    print('\nNEGATIVE side:')
    for idx, sim in zip(top_neg.indices.tolist(), top_neg.values.tolist()):
        desc = _fetch(idx)
        print(f'  feat {idx:6d}  cos={sim:+.3f}  {desc}')
        time.sleep(0.3)
else:
    print('Skipped (FETCH_AUTOINTERP=False). Click the URLs above for descriptions.')

## 6 — Sanity: random direction baseline

A random unit vector projected onto the SAE will produce some high-cosine features by chance. Compare your top-K cosines against a random baseline to confirm the projection isn't noise.

In [ ]:
torch.manual_seed(42)
rand = torch.randn(d_model)
rand = rand / rand.norm()
rand_sims = (W_dec @ rand) / dec_norms
rand_top = torch.topk(rand_sims, TOP_K, largest=True)

print(f'YOUR direction         top-{TOP_K} max  cos_dec = {top_pos.values[0]:+.4f}')
print(f'random unit vector     top-{TOP_K} max  cos_dec = {rand_top.values[0]:+.4f}')
print(f'\nYOUR direction         top-{TOP_K} mean cos_dec = {top_pos.values.mean():+.4f}')
print(f'random unit vector     top-{TOP_K} mean cos_dec = {rand_top.values.mean():+.4f}')
print(f'\nratio (your max / random max) = {(top_pos.values[0] / rand_top.values[0]).item():.2f}x')
print('\n(Ratio >> 1 means the direction is genuinely picking up structured features.\n'
      ' Ratio ~ 1 means top features are no better than chance — direction may not be meaningful here.)')